#  Steelproof Temperature Prediction

##  Objective
Predict the final temperature of molten steel to optimize energy consumption.

## Data Sources
Multiple datasets: arc, gas, bulk materials, wire, temperature.

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

RANDOM_STATE = 42

In [5]:
arc = pd.read_csv('../data/data_arc_en.csv')
temp = pd.read_csv('../data/data_temp_en.csv')
gas = pd.read_csv('../data/data_gas_en.csv')
bulk = pd.read_csv('../data/data_bulk_en.csv')
wire = pd.read_csv('../data/data_wire_en.csv')

##  Data Preparation

- Aggregation by batch (`key`)
- Creation of target variable
- Removal of non-numeric features

In [8]:
target = temp.groupby('key')['Temperature'].last()

In [9]:
arc_agg = arc.groupby('key').sum()
gas_agg = gas.groupby('key').sum()
bulk_agg = bulk.groupby('key').sum()
wire_agg = wire.groupby('key').sum()

In [10]:
data = target.to_frame().join(
    [arc_agg, gas_agg, bulk_agg, wire_agg],
    how='inner'
)

data.head()

,Temperature,Arc heating start,Arc heating end,Active power,Reactive power,Gas 1,Bulk 1,Bulk 2,Bulk 3,Bulk 4,...,Bulk 15,Wire 1,Wire 2,Wire 3,Wire 4,Wire 5,Wire 6,Wire 7,Wire 8,Wire 9
key,,,,,,,,,,,,,,,,,,,,,
1,1613.0,2019-05-03 11:02:142019-05-03 11:07:282019-05-...,2019-05-03 11:06:022019-05-03 11:10:332019-05-...,4.878147,3.183241,29.749986,0.0,0.0,0.0,43.0,...,154.0,60.059998,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1602.0,2019-05-03 11:34:142019-05-03 11:38:502019-05-...,2019-05-03 11:36:312019-05-03 11:44:282019-05-...,3.052598,1.998112,12.555561,0.0,0.0,0.0,73.0,...,154.0,96.052315,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1599.0,2019-05-03 12:06:542019-05-03 12:13:522019-05-...,2019-05-03 12:11:342019-05-03 12:15:562019-05-...,2.525882,1.599076,28.554793,0.0,0.0,0.0,34.0,...,153.0,91.160157,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1625.0,2019-05-03 12:39:372019-05-03 12:44:472019-05-...,2019-05-03 12:43:042019-05-03 12:46:262019-05-...,3.209250,2.060298,18.841219,0.0,0.0,0.0,81.0,...,154.0,89.063515,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1602.0,2019-05-03 13:11:132019-05-03 13:18:212019-05-...,2019-05-03 13:15:242019-05-03 13:20:332019-05-...,3.347173,2.252643,5.413692,0.0,0.0,0.0,78.0,...,152.0,89.238236,9.11456,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [37]:
data = target.to_frame().join(
    [arc_agg, gas_agg, bulk_agg, wire_agg],
    how='inner'
)

data = data.select_dtypes(include=[np.number])

data.head()

,Temperature,Active power,Reactive power,Gas 1,Bulk 1,Bulk 2,Bulk 3,Bulk 4,Bulk 5,Bulk 6,...,Bulk 15,Wire 1,Wire 2,Wire 3,Wire 4,Wire 5,Wire 6,Wire 7,Wire 8,Wire 9
key,,,,,,,,,,,,,,,,,,,,,
1,1613.0,4.878147,3.183241,29.749986,0.0,0.0,0.0,43.0,0.0,0.0,...,154.0,60.059998,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1602.0,3.052598,1.998112,12.555561,0.0,0.0,0.0,73.0,0.0,0.0,...,154.0,96.052315,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1599.0,2.525882,1.599076,28.554793,0.0,0.0,0.0,34.0,0.0,0.0,...,153.0,91.160157,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1625.0,3.209250,2.060298,18.841219,0.0,0.0,0.0,81.0,0.0,0.0,...,154.0,89.063515,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1602.0,3.347173,2.252643,5.413692,0.0,0.0,0.0,78.0,0.0,0.0,...,152.0,89.238236,9.11456,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
data = data.select_dtypes(include=[np.number])

In [22]:
X = data.drop(columns='Temperature')
y = data['Temperature']

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

## 🤖 Modeling

We compare a baseline model (DummyRegressor) with a Random Forest model.

In [31]:
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Dummy MAE:", mean_absolute_error(y_test, y_pred_dummy))

Dummy MAE: 11.162250176949083


In [29]:
rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(mean_absolute_error(y_test, y_pred_rf))

10.050057851239671


In [33]:
print("Random Forest Results:")
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", mean_squared_error(y_test, y_pred_rf
                                  ))
print("R2:", r2_score(y_test, y_pred_rf))

Random Forest Results:
MAE: 10.050057851239671
RMSE: 196.40670607438025
R2: 0.206755042055637


In [34]:
print("\nDummy Model Results:")
print("MAE:", mean_absolute_error(y_test, y_pred_dummy))


Dummy Model Results:
MAE: 11.162250176949083


In [36]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

importances.head(10)

Gas 1             0.205396
Wire 1            0.136990
Active power      0.095396
Reactive power    0.082551
Bulk 14           0.076138
Bulk 15           0.074295
Bulk 12           0.062945
Bulk 4            0.058061
Wire 2            0.051150
Bulk 3            0.041171
dtype: float64

## 📈 Conclusions

- Random Forest outperformed the baseline model
- The process has nonlinear relationships
- Machine learning can help optimize energy consumption

## ⚠️ Limitations

- Lack of time-based features
- Possible missing variables affecting temperature